In [ ]:
pip install ultralytics opencv-python numpy

In [ ]:
import cv2
import numpy as np

# --------------- CONFIG ----------------

VIDEO_PATH       = "vid1.avi"
OUTPUT_PATH      = "vid1_Clip.mp4"

# How much clip to keep around the chosen event
PRE_EVENT_SEC        = 6.0   # seconds BEFORE the (adjusted) event
POST_EVENT_SEC       = 6.0   # seconds AFTER the (adjusted) event

# For ~2s falls, shift the event ~1s earlier so we see the start of the fall
EARLY_FALL_OFFSET_SEC = 1.0  # how many seconds earlier than detection center

# Motion thresholds (tune these based on your videos)
FALL_THRESHOLD   = 0.8       # 95th percentile motion for fall frames
FALL_MIN_FRAMES  = 3         # consecutive sampled frames above FALL_THRESHOLD

HIT_MINOR_THRESHOLD  = 0.7   # anything above this is at least a "hit"
HIT_MAJOR_THRESHOLD  = 1.2   # above this → "major hit"

# Speed / quality knobs
DETECTION_SCALE  = 0.5       # 0.5 = process at 50% resolution
FRAME_STEP       = 2         # analyze every Nth frame only
DEBUG            = False     # True → print per-frame motion score
# --------------------------------------


def detect_fall_or_hit_and_extract_clip(
    video_path: str,
    output_path: str,
    fall_threshold: float,
    fall_min_frames: int,
    hit_minor_threshold: float,
    hit_major_threshold: float,
):
    # ---------- PASS 1: FAST ANALYSIS (DETECTION) ----------
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not open video.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Read first frame
    ret, frame = cap.read()
    if not ret:
        print("Error: Could not read first frame.")
        cap.release()
        return

    # Prepare first low-res grayscale frame
    small_prev = cv2.resize(
        frame, None,
        fx=DETECTION_SCALE,
        fy=DETECTION_SCALE,
        interpolation=cv2.INTER_AREA,
    )
    prev_gray = cv2.cvtColor(small_prev, cv2.COLOR_BGR2GRAY)

    # ---- tracking for FALL (sustained high motion) ----
    in_fall_run = False
    run_start_frame = None
    run_scores = []

    best_fall_severity = 0.0
    best_fall_center_frame = None
    best_fall_run_len = 0
    best_fall_run_avg = 0.0

    # ---- tracking for HIT (single sharp spike) ----
    max_hit_score = 0.0
    max_hit_frame = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_pos = int(cap.get(cv2.CAP_PROP_POS_FRAMES))  # current real frame index

        # Only analyze every Nth frame
        if frame_pos % FRAME_STEP != 0:
            continue

        # Downscale frame for speed
        small = cv2.resize(
            frame, None,
            fx=DETECTION_SCALE,
            fy=DETECTION_SCALE,
            interpolation=cv2.INTER_AREA,
        )
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, gray, None,
            0.5,   # pyr_scale
            1,     # levels
            9,     # winsize
            2,     # iterations
            5,     # poly_n
            1.1,   # poly_sigma
            0      # flags
        )

        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        # Use 95th percentile so local strong motion (like one leg) counts
        motion_score = np.percentile(mag, 95)

        if DEBUG:
            print(f"frame {frame_pos:5d} | motion_score={motion_score:.3f}")

        # ---------- FALL LOGIC: sustained motion above threshold ----------
        if motion_score > fall_threshold:
            if not in_fall_run:
                in_fall_run = True
                run_start_frame = frame_pos
                run_scores = [motion_score]
            else:
                run_scores.append(motion_score)
        else:
            # run ended → evaluate it
            if in_fall_run and len(run_scores) >= fall_min_frames:
                run_len = len(run_scores)
                run_avg = float(np.mean(run_scores))
                # severity combines strength and duration
                run_severity = run_avg * run_len

                if run_severity > best_fall_severity:
                    best_fall_severity = run_severity
                    best_fall_run_len = run_len
                    best_fall_run_avg = run_avg
                    # approximate center frame of this fall run
                    offset_frames = (run_len // 2) * FRAME_STEP
                    best_fall_center_frame = run_start_frame + offset_frames

            # reset run state
            in_fall_run = False
            run_start_frame = None
            run_scores = []

        # ---------- HIT LOGIC: track largest spike ----------
        if motion_score > max_hit_score:
            max_hit_score = motion_score
            max_hit_frame = frame_pos

        prev_gray = gray

    # Handle if video ended while still in a fall run
    if in_fall_run and len(run_scores) >= fall_min_frames:
        run_len = len(run_scores)
        run_avg = float(np.mean(run_scores))
        run_severity = run_avg * run_len
        if run_severity > best_fall_severity:
            best_fall_severity = run_severity
            best_fall_run_len = run_len
            best_fall_run_avg = run_avg
            offset_frames = (run_len // 2) * FRAME_STEP
            best_fall_center_frame = run_start_frame + offset_frames

    cap.release()

    # ---------- CLASSIFY EVENTS ----------
    fall_detected = best_fall_center_frame is not None
    hit_detected  = (max_hit_frame is not None and max_hit_score >= hit_minor_threshold)
    major_hit     = (max_hit_score >= hit_major_threshold)

    if DEBUG:
        print("---- SUMMARY ----")
        print(f"Fall_detected: {fall_detected}, "
              f"best_fall_center_frame={best_fall_center_frame}, "
              f"fall_avg={best_fall_run_avg:.3f}, "
              f"fall_len={best_fall_run_len}, "
              f"fall_severity={best_fall_severity:.3f}")
        print(f"Hit_detected: {hit_detected}, "
              f"max_hit_frame={max_hit_frame}, "
              f"max_hit_score={max_hit_score:.3f}, "
              f"major_hit={major_hit}")

    event_frame_idx = None
    event_type = None

    # ---- DECISION PRIORITY ----
    # 1) If fall exists and hit is NOT major → prefer fall (your requirement)
    if fall_detected and (not major_hit):
        event_frame_idx = best_fall_center_frame
        event_type = "fall"

    # 2) Else if fall exists and hit is major → still prefer fall by default
    elif fall_detected and major_hit:
        event_frame_idx = best_fall_center_frame
        event_type = "fall"

    # 3) Else if no fall, but hit exists → use hit
    elif hit_detected:
        event_frame_idx = max_hit_frame
        event_type = "hit_major" if major_hit else "hit_minor"

    else:
        print("No fall or hit detected. "
              f"Max motion_score={max_hit_score:.3f} "
              f"(try lowering thresholds).")
        return

    print(f"Chosen event type: {event_type}, at frame {event_frame_idx}")
    if fall_detected:
        print(f"  Best fall: center={best_fall_center_frame}, "
              f"avg={best_fall_run_avg:.3f}, len={best_fall_run_len}, "
              f"severity={best_fall_severity:.3f}")
    if hit_detected:
        print(f"  Hit: frame={max_hit_frame}, score={max_hit_score:.3f}, "
              f"major={major_hit}")

    # ---------- PASS 2: TRIM ORIGINAL VIDEO AROUND CHOSEN EVENT ----------
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not reopen video for trimming.")
        return

    # Use FPS from the file again (safe) for timing-based offsets
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Shift fall a bit earlier so we see the start of the fall
    if event_type.startswith("fall"):
        early_frames = int(EARLY_FALL_OFFSET_SEC * fps)
        event_frame_idx_adjusted = max(0, int(event_frame_idx) - early_frames)
    else:
        event_frame_idx_adjusted = int(event_frame_idx)

    # Convert pre/post seconds into frame offsets
    frames_before = int(PRE_EVENT_SEC * fps)
    frames_after  = int(POST_EVENT_SEC * fps)

    start_frame = max(0, event_frame_idx_adjusted - frames_before)
    end_frame   = min(total_frames - 1, event_frame_idx_adjusted + frames_after)

    frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(
        output_path, fourcc, fps, (frame_width, frame_height)
    )

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    while True:
        frame_pos = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        if frame_pos > end_frame:
            break

        ret, frame = cap.read()
        if not ret:
            break

        out.write(frame)

    cap.release()
    out.release()

    duration_sec = (end_frame - start_frame) / fps if fps > 0 else 0
    print(
        f"{event_type.upper()} clip saved as: {output_path}\n"
        f"Frames: {start_frame}–{end_frame}, "
        f"duration ~{duration_sec:.2f}s"
    )


# Run
if __name__ == "__main__":
    detect_fall_or_hit_and_extract_clip(
        VIDEO_PATH,
        OUTPUT_PATH,
        FALL_THRESHOLD,
        FALL_MIN_FRAMES,
        HIT_MINOR_THRESHOLD,
        HIT_MAJOR_THRESHOLD,
    )


Chosen event type: fall, at frame 474
  Best fall: center=474, avg=1.198, len=38, severity=45.535
  Hit: frame=486, score=1.757, major=True
FALL clip saved as: cam8_NoFallClip.mp4
Frames: 0–1074, duration ~8.95s
